# 🤖 Digital Twin Chatbot

A personal AI assistant that represents **you** — powered by Claude (Anthropic) + Gradio.

**Features:**
- Answers questions about your background, skills, and experience
- Naturally collects visitor name + email when the conversation is going well
- Logs unanswered questions for your review
- Agentic tool-calling loop (same pattern as the original OpenAI version)

---
**To use:** Edit your details in **Cell 2**, add your API key in **Cell 3**, then **Run All**.

In [ ]:
%pip install anthropic gradio --quiet

## ⚙️ Configure Your Digital Twin
Replace the placeholder text below with your own information.

In [ ]:
# ============================================================
#  YOUR DETAILS — edit everything in this cell
# ============================================================

NAME = "Alex Rivera"

KNOWLEDGE_BASE = """
EXPERIENCE:
- 8 years of full-stack engineering (React, Node.js, Python, Go)
- Led a team of 6 engineers at Vanta (security compliance SaaS) 2021-2024
- Built the core data pipeline at Mosaic (fintech) handling $2B+ in transactions
- Early engineer at two YC-backed startups (both acquired)
- Open source maintainer of 'fastqueue' — a job queue library with 4k GitHub stars

SKILLS:
- Frontend: React, Next.js, TypeScript, Tailwind, WebSockets
- Backend: Node.js, Python/FastAPI, Go, PostgreSQL, Redis, Kafka
- Infra: AWS, GCP, Terraform, Docker, Kubernetes
- Leadership: hiring, technical roadmaps, cross-functional alignment

EDUCATION:
- B.S. Computer Science, University of Michigan, 2016

PROJECTS:
- ShipFast: a SaaS boilerplate used by 2,000+ founders
- Local-first notes app with CRDT sync (open source)

INTERESTS:
- Distributed systems, developer tooling, building in public
- Rock climbing, specialty coffee, sci-fi novels

AVAILABILITY:
- Open to new roles starting Q3 2025
- Prefer remote-first; open to hybrid in SF Bay Area
- Not looking for purely management roles — want to stay hands-on technically
"""

# ============================================================

## 🔑 API Key

In [ ]:
import os

# Option A: paste key directly (fine for local use)
os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"

# Option B: prompt securely (uncomment to use)
# import getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")

## 🧠 DigitalTwin Class

In [ ]:
import json
import anthropic
from typing import List, Dict


class DigitalTwin:
    """
    An AI persona that represents a real person in a Gradio chat interface.

    Uses Claude with tool-calling to:
      - Answer questions from an inline knowledge base
      - Collect visitor contact details (name + email)
      - Log questions it couldn't answer
    """

    MODEL = "claude-opus-4-5"

    def __init__(self, name: str, knowledge_base: str):
        self.name = name
        self.knowledge_base = knowledge_base
        self.client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

        # Contact state
        self.contact_collected = False
        self.user_name: str | None = None
        self.user_email: str | None = None

        # In-memory logs (inspect in Cell 8)
        self.contacts_log: List[Dict] = []
        self.unknown_questions_log: List[str] = []

        # Tool schemas
        self.tools = [
            {
                "name": "record_user_details",
                "description": (
                    "Save a visitor's contact info. "
                    "Only call this after you have BOTH their name AND email. "
                    "If they only gave an email, ask for their name first."
                ),
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "email": {"type": "string", "description": "Visitor's email address"},
                        "name":  {"type": "string", "description": "Visitor's full name"},
                        "notes": {"type": "string", "description": "One-line summary of their interest"}
                    },
                    "required": ["email", "name", "notes"]
                }
            },
            {
                "name": "record_unknown_question",
                "description": "Log a question you couldn't answer so it can be reviewed later.",
                "input_schema": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "The unanswered question"}
                    },
                    "required": ["question"]
                }
            }
        ]

        print(f"Digital Twin for '{self.name}' ready.")

    # ------------------------------------------------------------------
    # Tool implementations
    # ------------------------------------------------------------------

    def record_user_details(self, email: str, name: str, notes: str) -> Dict:
        self.contact_collected = True
        self.user_name  = name
        self.user_email = email
        self.contacts_log.append({"name": name, "email": email, "notes": notes})
        print(f"[CONTACT] {name} <{email}> | {notes}")
        return {"recorded": "ok", "message": f"Thanks {name}, I'll be in touch!"}

    def record_unknown_question(self, question: str) -> Dict:
        self.unknown_questions_log.append(question)
        print(f"[UNKNOWN] {question}")
        return {"recorded": "ok", "message": "Noted — I'll look into that."}

    def _dispatch_tool(self, name: str, inputs: Dict) -> Dict:
        fn = getattr(self, name, None)
        return fn(**inputs) if fn else {"error": f"Unknown tool: {name}"}

    # ------------------------------------------------------------------
    # System prompt (rebuilt each turn so contact state stays current)
    # ------------------------------------------------------------------

    def _system_prompt(self) -> str:
        p = (
            f"You are acting as {self.name}. You are on {self.name}'s personal website "
            f"answering questions from visitors — potential employers, collaborators, or clients.\n\n"
            f"## Knowledge Base\n{self.knowledge_base}\n\n"
            "## Tone & Style\n"
            "- Be direct, warm, and genuine — not corporate or stiff.\n"
            "- Be specific; generic answers are boring.\n"
            "- If you truly don't know something, call record_unknown_question and admit it honestly.\n"
        )
        if not self.contact_collected:
            p += (
                "\n## Contact Collection\n"
                "- If the conversation is going well, naturally invite the visitor to stay in touch.\n"
                "- Collect BOTH their name and email before calling record_user_details.\n"
                "- Ask for name first if they only share an email.\n"
            )
        else:
            p += (
                f"\nYou already have contact details for {self.user_name} ({self.user_email}). "
                "Do NOT ask for contact details again.\n"
            )
        p += f"\nAlways stay in character as {self.name}."
        return p

    # ------------------------------------------------------------------
    # Agentic chat loop
    # ------------------------------------------------------------------

    def chat(self, message: str, history: List) -> str:
        """Gradio-compatible chat function."""
        # Convert Gradio history → Anthropic message list
        messages: List[Dict] = []
        for turn in history:
            if isinstance(turn, (list, tuple)) and len(turn) == 2:
                u, b = turn
                if u: messages.append({"role": "user",      "content": str(u)})
                if b: messages.append({"role": "assistant", "content": str(b)})
            elif isinstance(turn, dict) and turn.get("role") in ("user", "assistant"):
                messages.append({"role": turn["role"], "content": str(turn["content"])})

        messages.append({"role": "user", "content": message})

        # Agentic loop — iterate until Claude stops calling tools
        for _ in range(5):
            response = self.client.messages.create(
                model=self.MODEL,
                max_tokens=1024,
                system=self._system_prompt(),
                tools=self.tools,
                messages=messages
            )

            if response.stop_reason == "tool_use":
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        print(f"[TOOL] {block.name}({block.input})")
                        result = self._dispatch_tool(block.name, block.input)
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": json.dumps(result)
                        })
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user",      "content": tool_results})
            else:
                for block in response.content:
                    if hasattr(block, "text"):
                        return block.text
                return "(no response)"

        return "(max iterations reached)"

## 🚀 Initialise

In [ ]:
twin = DigitalTwin(name=NAME, knowledge_base=KNOWLEDGE_BASE)

## 🧪 Quick Test (optional)
Send a single message without launching the full UI.

In [ ]:
reply = twin.chat("What tech stack do you work with?", history=[])
print(reply)

## 💬 Launch Gradio Chat UI
Opens on `http://localhost:7860` by default.

In [ ]:
import gradio as gr


def chat_wrapper(message, history):
    return twin.chat(message, history)


with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
    css="#chatbot { height: 520px; } .contain { max-width: 860px; margin: auto; }"
) as demo:

    gr.Markdown(f"""
# 💬 Chat with {twin.name}
I'm an AI assistant representing {twin.name}.
Ask me anything about my background, experience, skills, or what I'm working on.
""")

    gr.ChatInterface(
        fn=chat_wrapper,
        chatbot=gr.Chatbot(elem_id="chatbot", bubble_full_width=False),
        textbox=gr.Textbox(
            placeholder=f"Ask about {twin.name}'s experience, skills, availability...",
            container=False,
            scale=7
        ),
        examples=[
            "What are you working on right now?",
            "What kind of role are you looking for?",
            "Tell me about your biggest project.",
            "What's your tech stack?",
        ],
        title=None,
        description=None,
    )

    gr.Markdown("---\n*Powered by Claude (Anthropic)*")


demo.launch(share=False, server_name="0.0.0.0", server_port=7860)

## 📋 View Logs
Run this cell any time to inspect collected contacts and unanswered questions.

In [ ]:
import pandas as pd

print("=== Collected Contacts ===")
if twin.contacts_log:
    display(pd.DataFrame(twin.contacts_log))
else:
    print("None yet.")

print("\n=== Unanswered Questions ===")
if twin.unknown_questions_log:
    for i, q in enumerate(twin.unknown_questions_log, 1):
        print(f"  {i}. {q}")
else:
    print("None yet.")